# Advanced Problems with Solutions: Python Decorators

Topic: decorators, closures, `*args` / `**kwargs`, metadata preservation, `functools.wraps`, introspection, and decorator composition.

Each problem includes a full solution.

## Problem 1 — Build a metadata-preserving call counter

Write a decorator `count_calls` that:

- counts how many times the decorated function is called,
- returns the original function result,
- preserves `__name__`, `__doc__`, annotations, and inspectable signature,
- exposes the current count as `function.call_count()`.

In [1]:
from functools import wraps
import inspect

def count_calls(fn):
    count = 0

    @wraps(fn)
    def wrapper(*args, **kwargs):
        nonlocal count
        count += 1
        return fn(*args, **kwargs)

    def call_count():
        return count

    wrapper.call_count = call_count
    return wrapper

@count_calls
def add(a: int, b: int = 10) -> int:
    """Return the sum of a and b."""
    return a + b

assert add(1, 2) == 3
assert add(5) == 15
assert add.call_count() == 2
assert add.__name__ == "add"
assert add.__doc__ == "Return the sum of a and b."
assert str(inspect.signature(add)) == "(a: int, b: int = 10) -> int"

print("All tests passed.")

All tests passed.


### Solution explanation

`count` lives in the decorator closure. The wrapper uses `nonlocal count` so it can update the enclosed variable. `@wraps(fn)` copies important metadata from the original function and sets `__wrapped__`, which allows tools such as `inspect.signature` to recover the original signature.

## Problem 2 — Debug a broken decorator

The following decorator is intended to print the function name and return the function result, but it has two problems:

```python
def debug(fn):
    def wrapper(*args, **kwargs):
        print(f'Calling {fn.__name__}')
        fn(*args, **kwargs)
    return wrapper
```

Fix it so that it:

- returns the original result,
- preserves metadata,
- works with positional and keyword arguments.

In [2]:
from functools import wraps

def debug(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        print(f"Calling {fn.__name__}")
        return fn(*args, **kwargs)
    return wrapper

@debug
def power(base: int, exponent: int = 2) -> int:
    """Raise base to exponent."""
    return base ** exponent

assert power(3) == 9
assert power(2, exponent=5) == 32
assert power.__name__ == "power"
assert power.__doc__ == "Raise base to exponent."

print("All tests passed.")

Calling power
Calling power
All tests passed.


### Solution explanation

The original decorator called `fn(*args, **kwargs)` but did not return its result. That silently turned every decorated function into a function returning `None`. Adding `return` fixes the behavior. Adding `@wraps(fn)` fixes the metadata loss.

## Problem 3 — Create a parameterized retry decorator

Write a decorator factory `retry(max_attempts)`.

It should:

- retry a function up to `max_attempts` times,
- stop immediately once the function succeeds,
- re-raise the last exception if all attempts fail,
- preserve metadata using `wraps`.

In [3]:
from functools import wraps

def retry(max_attempts):
    if max_attempts < 1:
        raise ValueError("max_attempts must be at least 1")

    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except Exception as ex:
                    last_error = ex
                    print(f"Attempt {attempt} failed: {ex}")
            raise last_error
        return wrapper
    return decorator

state = {"calls": 0}

@retry(3)
def unstable_task():
    """Succeed on the third call."""
    state["calls"] += 1
    if state["calls"] < 3:
        raise RuntimeError("not ready")
    return "success"

assert unstable_task() == "success"
assert state["calls"] == 3
assert unstable_task.__name__ == "unstable_task"

print("All tests passed.")

Attempt 1 failed: not ready
Attempt 2 failed: not ready
All tests passed.


### Solution explanation

`retry(max_attempts)` is not itself the final decorator. It returns the real decorator, which then receives the function. This is the standard pattern for decorators that accept arguments.

## Problem 4 — Understand decorator stacking order

Create two decorators, `uppercase` and `surround`, then apply them to the same function.

Verify that:

```python
@decorator_a
@decorator_b
def fn():
    ...
```

is equivalent to:

```python
fn = decorator_a(decorator_b(fn))
```

In [4]:
from functools import wraps

def uppercase(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs).upper()
    return wrapper

def surround(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        return f"<<<{fn(*args, **kwargs)}>>>"
    return wrapper

@surround
@uppercase
def greet(name):
    """Return a greeting."""
    return f"hello, {name}"

assert greet("Ada") == "<<<HELLO, ADA>>>"
assert greet.__name__ == "greet"

def greet_manual(name):
    return f"hello, {name}"

greet_manual = surround(uppercase(greet_manual))

assert greet_manual("Ada") == greet("Ada")

print("All tests passed.")

All tests passed.


### Solution explanation

The decorator closest to the function is applied first. In this example, `uppercase` runs before `surround`, so the greeting is uppercased first and surrounded second.

## Problem 5 — Write a decorator that validates return type

Write a decorator `enforce_return_type` that checks the function's return annotation.

Requirements:

- If there is no return annotation, do nothing.
- If there is a return annotation, check that the returned value matches it.
- Raise `TypeError` when the result has the wrong type.
- Preserve metadata.

In [5]:
from functools import wraps
import inspect

def enforce_return_type(fn):
    signature = inspect.signature(fn)
    expected = signature.return_annotation

    @wraps(fn)
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)

        if expected is inspect.Signature.empty:
            return result

        if not isinstance(result, expected):
            raise TypeError(
                f"{fn.__name__} expected to return {expected.__name__}, "
                f"but returned {type(result).__name__}"
            )

        return result

    return wrapper

@enforce_return_type
def make_name(first: str, last: str) -> str:
    return f"{first} {last}"

@enforce_return_type
def bad_add(a: int, b: int) -> int:
    return str(a + b)

assert make_name("Ada", "Lovelace") == "Ada Lovelace"

try:
    bad_add(1, 2)
except TypeError as ex:
    print(ex)
else:
    raise AssertionError("Expected TypeError")

print("All tests passed.")

bad_add expected to return int, but returned str
All tests passed.


### Solution explanation

`inspect.signature(fn).return_annotation` gives access to the function's declared return type. This solution handles simple runtime-checkable types such as `int`, `str`, `float`, `list`, and custom classes.

## Problem 6 — Build a decorator-based plugin registry

Create a decorator `register(name)` that stores functions in a dictionary called `registry`.

Requirements:

- `@register("csv")` should register the decorated function under the key `"csv"`.
- The decorated function should remain callable normally.
- Duplicate names should raise `KeyError`.
- Metadata should be preserved.

In [6]:
from functools import wraps

registry = {}

def register(name):
    def decorator(fn):
        if name in registry:
            raise KeyError(f"A plugin named {name!r} is already registered")

        @wraps(fn)
        def wrapper(*args, **kwargs):
            return fn(*args, **kwargs)

        registry[name] = wrapper
        return wrapper
    return decorator

@register("csv")
def parse_csv(text):
    """Parse comma-separated values."""
    return [part.strip() for part in text.split(",")]

@register("pipe")
def parse_pipe(text):
    """Parse pipe-separated values."""
    return [part.strip() for part in text.split("|")]

assert parse_csv("a, b, c") == ["a", "b", "c"]
assert registry["csv"]("x, y") == ["x", "y"]
assert registry["pipe"]("x | y") == ["x", "y"]
assert parse_csv.__name__ == "parse_csv"

print("Registered plugins:", sorted(registry))
print("All tests passed.")

Registered plugins: ['csv', 'pipe']
All tests passed.


### Solution explanation

The decorator factory receives the registry key. The actual decorator receives the function. The wrapper is stored in the registry and returned so the function name still refers to a callable.

## Problem 7 — Preserve access to the undecorated function

Use `functools.wraps` and `__wrapped__` to prove that a decorated function still exposes the original undecorated function.

Create a decorator that blocks negative inputs, then call the undecorated original through `__wrapped__`.

In [7]:
from functools import wraps

def block_negative(fn):
    @wraps(fn)
    def wrapper(x):
        if x < 0:
            raise ValueError("negative values are blocked")
        return fn(x)
    return wrapper

@block_negative
def square(x):
    """Return x squared."""
    return x * x

assert square(4) == 16

try:
    square(-4)
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError")

# Bypass the decorator intentionally:
assert square.__wrapped__(-4) == 16

print("Original function:", square.__wrapped__)
print("All tests passed.")

Original function: <function square at 0x0000028D8D472840>
All tests passed.


### Solution explanation

`wraps` adds a `__wrapped__` attribute pointing to the original function. This is useful for introspection, testing, and advanced decorator composition.

## Problem 8 — Combine timing and counting decorators correctly

Create two decorators:

- `timer`: records how long a function takes,
- `counter`: counts how many times a function is called.

Apply both to a function and verify that metadata is preserved.

In [8]:
from functools import wraps
from time import perf_counter, sleep
import inspect

def timer(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        try:
            return fn(*args, **kwargs)
        finally:
            elapsed = perf_counter() - start
            wrapper.last_elapsed = elapsed
    wrapper.last_elapsed = None
    return wrapper

def counter(fn):
    count = 0

    @wraps(fn)
    def wrapper(*args, **kwargs):
        nonlocal count
        count += 1
        return fn(*args, **kwargs)

    wrapper.count = lambda: count
    return wrapper

@counter
@timer
def slow_add(a: int, b: int) -> int:
    """Slowly add two integers."""
    sleep(0.01)
    return a + b

assert slow_add(2, 3) == 5
assert slow_add(10, 20) == 30
assert slow_add.count() == 2
assert slow_add.__name__ == "slow_add"
assert slow_add.__doc__ == "Slowly add two integers."
assert str(inspect.signature(slow_add)) == "(a: int, b: int) -> int"

print("Call count:", slow_add.count())
print("Signature:", inspect.signature(slow_add))
print("All tests passed.")

Call count: 2
Signature: (a: int, b: int) -> int
All tests passed.


### Solution explanation

Both decorators use `wraps`, so the final decorated function still looks like the original function to introspection tools. Because `counter` is the outer decorator, its custom `.count()` attribute is directly available on `slow_add`.